**Importing Libraries**

In [17]:
import pandas as pd
import numpy as np

**Loading Data**

In [18]:
# providing a base URL to our Data folder and then specific directions to each individual dataset file
# Omitting "Inventory and Sales" for right now because it is a different file type
base_url = "https://raw.githubusercontent.com/junseo25/ds2002-walmart-hurricane-project/main/data"

transactions = pd.read_csv(f"{base_url}/walmart_transactions.csv")

product = pd.read_csv(f"{base_url}/product_catalog.csv")

store = pd.read_csv(f"{base_url}/store_locations.csv")

distribution = pd.read_csv(f"{base_url}/distribution_centers.csv")



**Defining Problems in the Data**

In [19]:
#Creating a dictionary first
datasets = {
    "transactions": transactions,
    "product": product,
    "store": store,
    "distribution": distribution
}

# defining rows, duplicates, and missing values for all four files
for name, data in datasets.items():
    print(f"Dataset: {name}")
    print(f"Rows: {len(data)}")
    print(f"Duplicates: {data.duplicated().sum()}")
    print(f"Missing Values: {data.isnull().sum().sum()}")

Dataset: transactions
Rows: 25562
Duplicates: 30
Missing Values: 886
Dataset: product
Rows: 45
Duplicates: 0
Missing Values: 18
Dataset: store
Rows: 14
Duplicates: 0
Missing Values: 2
Dataset: distribution
Rows: 5
Duplicates: 0
Missing Values: 1


**Organizing the Product catalog first**

In [20]:
# Define products_clean by creating a copy of the raw product DataFrame
products_clean = product.copy()
products_clean["original_sku"] = products_clean["sku"]

In [21]:

# Standardizing product formatting while replacing underscores with spaces, preserving word separations, and capitalizing
products_clean['product_name'] = (
    products_clean['product_name']
    .str.lower()
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('_', ' ', regex=False)  # Replaces underscores with a single space
    .str.replace('pk', '-pack', regex=False)
    .str.replace('sz', 'size', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace(r'\s+', ' ', regex=True)  # Condenses any multiple spaces into a single space
    .str.strip()
    .str.title()
    .str.replace('Aa', 'AA', regex=False)  # 2. Convert 'Aa' to 'AA' last so title() doesn't overwrite it
)

#Creating a product map - Logic: Put the Product Type first, then quantity (if applicable)
PRODUCT_MAP = {'Strawberry Pop-Tart': 'Pop-Tarts Strawberry',
            'Strawberry Pop-Tart': 'Pop-Tarts Strawberry',
            'Water - 24 Pack': 'Bottled Water 24-Pack',
            'Batteries AA 4-Pack': 'AA Batteries 4-Pack',
            'Board Game - Monopoly': 'Monopoly Board Game',
            'Streaming Device': 'Roku Streaming Stick',
            'Soup - Canned': 'Canned Soup',
            'Portable Phone Charger': 'Portable Charger',
            'Variety Pack - Chips': 'Chips Variety Pack',
            'Toilet Paper 12-Roll': 'Toilet Paper 12-Pack',
            'Baby Diapers - Size 3': 'Diapers Size 3',
            'Infant Formula': 'Baby Formula',
            'Tarp - Heavy Duty': 'Heavy Duty Tarp',
            'First-Aid Kit': 'First Aid Kit',
            'Flashlight Led': 'LED Flashlight',
            'Candles Emergency': 'Emergency Candles'
}
#applying the mapping
products_clean["product_name"] = (
    products_clean["product_name"].replace(PRODUCT_MAP)
)
print(products_clean["product_name"].unique())


['Pop-Tarts Strawberry' 'Bottled Water 24-Pack' 'AA Batteries 4-Pack'
 'LED Flashlight' 'Light Beer 24-Pack' 'Monopoly Board Game'
 'Playing Cards' 'Roku Streaming Stick' 'Canned Soup' 'White Bread'
 'Peanut Butter' 'Emergency Candles' 'First Aid Kit' 'Portable Charger'
 'Chips Variety Pack' 'Paper Towels' 'Toilet Paper 12-Pack'
 'Diapers Size 3' 'Baby Formula' 'Heavy Duty Tarp']


In [22]:
# 1. Create a clean dictionary mapping standardized product names to their correct suppliers
PRODUCT_TO_SUPPLIER = {
    'Pop-Tarts Strawberry': "Kellogg's",
    'Bottled Water 24-Pack': 'Nestle',
    'AA Batteries 4-Pack': 'Duracell',
    'Flashlight Led': 'Everactive',
    'Light Beer 24-Pack': 'Anheuser-Busch',
    'Monopoly Board Game': 'Hasbro',
    'Playing Cards': 'Bicycle',
    'Roku Streaming Stick': 'Roku Inc',
    'Canned Soup': "Campbell's",
    'White Bread': 'Sara Lee',
    'Peanut Butter': 'J.M. Smucker',
    'Emergency Candles': 'Sterno',
    'First Aid Kit': 'Johnson & Johnson',
    'Portable Charger': 'Anker',
    'Chips Variety Pack': 'Frito-Lay',
    'Paper Towels 6-Pack': 'Georgia-Pacific',
    'Toilet Paper 12-Pack': 'Kimberly-Clark',
    'Diapers Size 3': 'Procter & Gamble',
    'Baby Formula': 'Abbott',
    'Heavy Duty Tarp': 'Tarps Plus'
}

# 2. Map the correct supplier onto the products_clean DataFrame using product_name
products_clean['supplier'] = products_clean['product_name'].map(PRODUCT_TO_SUPPLIER).fillna(products_clean['supplier'])

# 3. Display the standardized mapping table
supplier_mapping_df = products_clean[['product_name', 'supplier']].drop_duplicates().sort_values('product_name')
display(supplier_mapping_df)

,product_name,supplier
7,AA Batteries 4-Pack,Duracell
41,Baby Formula,Abbott
4,Bottled Water 24-Pack,Nestle
21,Canned Soup,Campbell's
33,Chips Variety Pack,Frito-Lay
39,Diapers Size 3,Procter & Gamble
27,Emergency Candles,Sterno
29,First Aid Kit,Johnson & Johnson
43,Heavy Duty Tarp,Tarps Plus
10,LED Flashlight,Frito-Lay


In [23]:
# Moving onto a SKU map - logic: give the most info possible in each SKU
SKU_MAP = {'PT-12': 'POPTART-STRAW',
           'SKU#459812': 'POPTART-STRAW',
           'PT_STRAWBERRY': 'POPTART-STRAW',
           'WTR-24': 'WATER24PK',
           'WTR_24PK': 'WATER24PK',
           'BATT-100': 'BATT_AA_4PK',
           'BAT-AA4': 'BATT_AA_4PK',
           'FLASH-01': 'LED-FLASH',
           'BEER-24': 'LTBEER24',
           'BG-MONO': 'GAME-MONO',
           'MONOPOLY-01': 'GAME-MONO',
           'PLAY-CARD': 'PLAYCARD',
           'CARDS-01': 'PLAYCARD',
           'PLAYCARD-STD': 'PLAYCARD',
           'ROKU-STD': 'ROKUSTREAM',
           'STREAM-01': 'ROKUSTREAM',
           'CNSOUP-01': 'SOUP-CAN',
           'BREAD-WH': 'WHITEBREAD',
           'PNUTBTR-16': 'PEANUTBTR',
           'PB-16': 'PEANUTBTR',
           'CNDL-EM': 'CANDLE-EMRG',
           'FAID-01': 'FIRSTAID-KIT',
           'CHRG-PORT': 'PORTCHARGER',
           'CHIPS-VP': 'CHIPVARIETY',
           'PTOWEL-6': 'PAPERTOWEL',
           'PAPERTOWEL6': 'PAPERTOWEL',
           'TP-12': 'TOILETPAPER12',
           'DIAP-S3': 'DIAPERS-S3',
           'DIAPERS-3': 'DIAPERS-S3',
           'BFORM-01': 'BABYFORMULA',
           'HDTARP-01': 'TARP-HD',
}

products_clean["canonical_sku"] = (
    products_clean["sku"].replace(SKU_MAP)
)

In [24]:
# first making sure unit_cost is numeric
products_clean['unit_cost'] = pd.to_numeric(products_clean['unit_cost'], errors='coerce')

# calculating the mean cost for each canonical product type
products_clean['unit_cost'] = products_clean.groupby('canonical_sku')['unit_cost'].transform('mean')

# Rounding the unit_cost to 2 decimal places
products_clean['unit_cost'] = products_clean['unit_cost'].round(2)

#Droping duplicate rows based on canonical_sku to keep only one unique entry per product type
products_clean = products_clean.drop_duplicates(subset=['canonical_sku']).reset_index(drop=True)

final_clean_catalog = products_clean[['canonical_sku', 'product_name', 'category', 'subcategory', 'supplier', 'package_size', 'unit_cost']]
display(final_clean_catalog)

,canonical_sku,product_name,category,subcategory,supplier,package_size,unit_cost
0,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast Foods,NaN,Kellogg's,4-pack,2.10
1,WATER24PK,Bottled Water 24-Pack,Emergency Supplies,Bottled,Nestle,1 pack,2.37
2,BATT_AA_4PK,AA Batteries 4-Pack,Emergency Supplies,AA,Duracell,6 pack,3.94
3,LED-FLASH,LED Flashlight,Emergency,Flashlight,Frito-Lay,6-pack,8.20
4,LTBEER24,Light Beer 24-Pack,Alcoholic Beverages,NaN,Anheuser-Busch,bundle,8.62
5,GAME-MONO,Monopoly Board Game,ENTERTAINMENT,NaN,Hasbro,1pk,11.24
6,PLAYCARD,Playing Cards,ENTERTAINMENT,Playing,Bicycle,4pk,2.42
7,ROKUSTREAM,Roku Streaming Stick,Electronics,Streaming,Roku Inc,12 pack,16.84
8,SOUP-CAN,Canned Soup,Groceries,Canned,Campbell's,single,1.21
9,WHITEBREAD,White Bread,Grocery,Bread,Sara Lee,24-pack,1.33


In [25]:
# organizing the cateory column - Logic: stripping unnecessary descriptions + standardizing capitalization conventions
products_clean['category'] = (
    products_clean['category']
    .str.lower()
    .str.title()
)
print(products_clean)
# creating a category map
CATEGORY_MAP = {'Breakfast Foods': 'Breakfast',
           'Emergency Supplies': 'Emergency',
           'Alcoholic Beverages': 'Alcohol',
           'Grocery': 'Groceries',
           'Baby': 'Baby Products',
}
products_clean['category'] = products_clean['category'].replace(CATEGORY_MAP)
print(products_clean)




          sku           product_name             category subcategory  \
0       PT-12   Pop-Tarts Strawberry      Breakfast Foods         NaN   
1      WTR-24  Bottled Water 24-Pack   Emergency Supplies     Bottled   
2    BATT-100    AA Batteries 4-Pack   Emergency Supplies          AA   
3    FLASH-01         LED Flashlight            Emergency  Flashlight   
4     BEER-24     Light Beer 24-Pack  Alcoholic Beverages         NaN   
5     BG-MONO    Monopoly Board Game        Entertainment         NaN   
6    CARDS-01          Playing Cards        Entertainment     Playing   
7   STREAM-01   Roku Streaming Stick          Electronics   Streaming   
8    SOUP-CAN            Canned Soup            Groceries      Canned   
9    BREAD-WH            White Bread              Grocery       Bread   
10      PB-16          Peanut Butter              Grocery      Peanut   
11    CNDL-EM      Emergency Candles            Emergency     Candles   
12    FAID-01          First Aid Kit               

In [26]:
products_clean.groupby("canonical_sku")["subcategory"].nunique()
products_clean["subcategory"] = (
    products_clean.groupby("canonical_sku")["subcategory"]
    .transform(lambda x: x.ffill().bfill())
)
print(products_clean)

          sku           product_name       category subcategory  unit_cost  \
0       PT-12   Pop-Tarts Strawberry      Breakfast         NaN       2.10   
1      WTR-24  Bottled Water 24-Pack      Emergency     Bottled       2.37   
2    BATT-100    AA Batteries 4-Pack      Emergency          AA       3.94   
3    FLASH-01         LED Flashlight      Emergency  Flashlight       8.20   
4     BEER-24     Light Beer 24-Pack        Alcohol         NaN       8.62   
5     BG-MONO    Monopoly Board Game  Entertainment         NaN      11.24   
6    CARDS-01          Playing Cards  Entertainment     Playing       2.42   
7   STREAM-01   Roku Streaming Stick    Electronics   Streaming      16.84   
8    SOUP-CAN            Canned Soup      Groceries      Canned       1.21   
9    BREAD-WH            White Bread      Groceries       Bread       1.33   
10      PB-16          Peanut Butter      Groceries      Peanut       1.65   
11    CNDL-EM      Emergency Candles      Emergency     Candles 

/tmp/ipykernel_812/3332099811.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.ffill().bfill())


In [27]:
#progress check

products_clean[
    ["original_sku", "canonical_sku", "product_name", "category", "subcategory"]
].head(25)

,original_sku,canonical_sku,product_name,category,subcategory
0,PT-12,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN
1,WTR-24,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled
2,BATT-100,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA
3,FLASH-01,LED-FLASH,LED Flashlight,Emergency,Flashlight
4,BEER-24,LTBEER24,Light Beer 24-Pack,Alcohol,NaN
5,BG-MONO,GAME-MONO,Monopoly Board Game,Entertainment,NaN
6,CARDS-01,PLAYCARD,Playing Cards,Entertainment,Playing
7,STREAM-01,ROKUSTREAM,Roku Streaming Stick,Electronics,Streaming
8,SOUP-CAN,SOUP-CAN,Canned Soup,Groceries,Canned
9,BREAD-WH,WHITEBREAD,White Bread,Groceries,Bread


In [28]:
# preserving missing values for costs at this stage
#moving on to suppliers, standardizing names

products_clean ["supplier"] = (
    products_clean["supplier"]
    .str.lower()
    .str.title()
    .str.replace(r"\s+", " ", regex=True)
)
print(products_clean)


          sku           product_name       category subcategory  unit_cost  \
0       PT-12   Pop-Tarts Strawberry      Breakfast         NaN       2.10   
1      WTR-24  Bottled Water 24-Pack      Emergency     Bottled       2.37   
2    BATT-100    AA Batteries 4-Pack      Emergency          AA       3.94   
3    FLASH-01         LED Flashlight      Emergency  Flashlight       8.20   
4     BEER-24     Light Beer 24-Pack        Alcohol         NaN       8.62   
5     BG-MONO    Monopoly Board Game  Entertainment         NaN      11.24   
6    CARDS-01          Playing Cards  Entertainment     Playing       2.42   
7   STREAM-01   Roku Streaming Stick    Electronics   Streaming      16.84   
8    SOUP-CAN            Canned Soup      Groceries      Canned       1.21   
9    BREAD-WH            White Bread      Groceries       Bread       1.33   
10      PB-16          Peanut Butter      Groceries      Peanut       1.65   
11    CNDL-EM      Emergency Candles      Emergency     Candles 

In [29]:
#standardizing how package sizes are written
products_clean["package_size"] = (
    products_clean["package_size"]
    .str.lower()
    .str.title()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace('Pk','Pack', regex=False)
    .str.replace('Single','1 Pack', regex=False)
    .str.replace('-', ' ', regex=False)
    # Insert a space between any number and 'Pack' (e.g., '24Pack' -> '24 Pack')
    .str.replace(r'(\d+)(Pack)', r'\1 \2', regex=True)
    .str.strip()
)
print(products_clean)

          sku           product_name       category subcategory  unit_cost  \
0       PT-12   Pop-Tarts Strawberry      Breakfast         NaN       2.10   
1      WTR-24  Bottled Water 24-Pack      Emergency     Bottled       2.37   
2    BATT-100    AA Batteries 4-Pack      Emergency          AA       3.94   
3    FLASH-01         LED Flashlight      Emergency  Flashlight       8.20   
4     BEER-24     Light Beer 24-Pack        Alcohol         NaN       8.62   
5     BG-MONO    Monopoly Board Game  Entertainment         NaN      11.24   
6    CARDS-01          Playing Cards  Entertainment     Playing       2.42   
7   STREAM-01   Roku Streaming Stick    Electronics   Streaming      16.84   
8    SOUP-CAN            Canned Soup      Groceries      Canned       1.21   
9    BREAD-WH            White Bread      Groceries       Bread       1.33   
10      PB-16          Peanut Butter      Groceries      Peanut       1.65   
11    CNDL-EM      Emergency Candles      Emergency     Candles 

In [30]:
#Progress so far

print(products_clean[["original_sku", "canonical_sku", "product_name", "category", "subcategory", "supplier", "package_size"]].head(25))

   original_sku  canonical_sku           product_name       category  \
0         PT-12  POPTART-STRAW   Pop-Tarts Strawberry      Breakfast   
1        WTR-24      WATER24PK  Bottled Water 24-Pack      Emergency   
2      BATT-100    BATT_AA_4PK    AA Batteries 4-Pack      Emergency   
3      FLASH-01      LED-FLASH         LED Flashlight      Emergency   
4       BEER-24       LTBEER24     Light Beer 24-Pack        Alcohol   
5       BG-MONO      GAME-MONO    Monopoly Board Game  Entertainment   
6      CARDS-01       PLAYCARD          Playing Cards  Entertainment   
7     STREAM-01     ROKUSTREAM   Roku Streaming Stick    Electronics   
8      SOUP-CAN       SOUP-CAN            Canned Soup      Groceries   
9      BREAD-WH     WHITEBREAD            White Bread      Groceries   
10        PB-16      PEANUTBTR          Peanut Butter      Groceries   
11      CNDL-EM    CANDLE-EMRG      Emergency Candles      Emergency   
12      FAID-01   FIRSTAID-KIT          First Aid Kit         He

**Cleaning the Store Data**

In [31]:
# Define stores_clean by creating a copy of the raw product DataFrame
stores_clean = store.copy()

In [32]:
#creating a function to standardize state code to be "FL-123"
def format_store_id(store_id):
    # Replace underscores
    store_id = store_id.replace('_', '-')

    # If the hyphen is already there, keep as is
    if '-' in store_id:
        return store_id

    # insert the hyphen after the first 2 state characters, if not present
    return f"{store_id[:2]}-{store_id[2:]}"

#Applying the function to the column
stores_clean = store.copy()
stores_clean['store_id'] = stores_clean['store_id'].apply(format_store_id)
print(stores_clean['store_id'].unique())

['FL-239' 'FL-105' 'FL-330' 'FL-412' 'FL-287' 'FL-156' 'FL-501' 'FL-078'
 'FL-619' 'FL-445' 'FL-302' 'FL-710']


In [33]:
# moving onto state name
def state_abbreviation(state):
    # Replace "Fla." with "FL"
    # Replacing "Florida" with "FL"
    state = state.replace('Fla.', 'FL')
    state = state.replace('Florida', 'FL')

    # If the hyphen is already there, keep as is
    if 'FL' in state:
        return state

#Applying the function to the column
stores_clean = store.copy()
stores_clean['state'] = stores_clean['state'].apply(state_abbreviation)
print(stores_clean['state'].unique())

#checking to make changes were implemented
print(stores_clean)

['FL']
   store_id                             store_name            city state  \
0    FL-239            Walmart Supercenter Orlando         Orlando    FL   
1    FL-105              Walmart Supercenter Tampa           Tampa    FL   
2    FL-330              Walmart Supercenter Miami           Miami    FL   
3    FL-412       Walmart Supercenter Jacksonville    Jacksonville    FL   
4    FL-287  Walmart Neighborhood Market Kissimmee       Kissimmee    FL   
5    FL-156      Walmart Supercenter Daytona Beach   Daytona Beach    FL   
6    FL-501         Walmart Supercenter Fort Myers      Fort Myers    FL   
7    FL-078        Walmart Supercenter Gainesville     Gainesville    FL   
8    FL-619        Walmart Supercenter Tallahassee     Tallahassee    FL   
9    FL-445     Walmart Supercenter St. Petersburg  St. Petersburg    FL   
10   FL-302           Walmart Supercenter Lakeland        Lakeland    FL   
11   FL-710          Walmart Supercenter Pensacola       Pensacola    FL   
12   

Distribution Center Data

In [34]:
# Define distribution_clean by creating a copy of the raw product DataFrame
distribution_clean = distribution.copy()

In [35]:
#creating a function to standardize surge_capacity to be just a number
def format_surge_capacity(surge_capacity):
    # Omit "~"
    surge_capacity = surge_capacity.replace('~', ' ')

    # If the "~" is not present, keep as is
    if '~' not in surge_capacity:
        return surge_capacity


#Applying the function to the column
distribution_clean = distribution.copy()
distribution_clean['surge_capacity'] = distribution_clean['surge_capacity'].apply(format_surge_capacity)
print (distribution_clean)

    dc_id                           dc_name          city state  \
0  DC-JAX  Jacksonville Distribution Center  Jacksonville    FL   
1  DC-ATL       Atlanta Distribution Center       Atlanta    GA   
2  DC-DAL        Dallas Distribution Center        Dallas    TX   
3  DC-MIA         Miami Distribution Center         Miami    FL   
4  DC-BHM    Birmingham Distribution Center    Birmingham    AL   

   avg_daily_shipments surge_capacity  distance_to_orlando_mi  \
0                  500           1000                     141   
1                  400            700                     438   
2                  300            600                    1108   
3                  350            750                     235   
4                  250            500                     503   

   cost_per_shipment  
0              245.0  
1              412.5  
2              687.0  
3              298.0  
4                NaN  


**Transaction Data**

In [36]:
# Define transactions_clean by creating a copy of the raw product DataFrame and then identifying duplicates and dropping them
transactions_clean = transactions.copy()

print("Before:", len(transactions_clean))

transactions_clean = transactions_clean.drop_duplicates()

print("After:", len(transactions_clean))



Before: 25562
After: 25532


In [37]:
# creating a function to ensure transaction_id is consistent
def format_transaction_id(transaction_id):
    if not isinstance(transaction_id, str):
        return transaction_id
    # If 'TXN' is already there, keep as is
    if 'TXN' in transaction_id:
        return transaction_id
    return transaction_id

# Applying the function directly to the existing transactions_clean
transactions_clean['transaction_id'] = transactions_clean['transaction_id'].apply(format_transaction_id)
print(transactions_clean['transaction_id'].unique())

['TXN012636' 'TXN000759' 'TXN003338' ... 'TXN002434' 'TXN014739'
 'TXN010981']


In [38]:
# function to ensure store code matches code established in store data 'FL-123'
def format_state_code(store_id):
    if not isinstance(store_id, str):
        return store_id

    # Remove leading/trailing whitespaces and convert to uppercase
    store_id = store_id.strip().upper()

    # Replace underscores
    store_id = store_id.replace('_', '-')

    # If the hyphen is already there, keep as is
    if '-' in store_id:
        return store_id

    # Insert the hyphen after the first 2 state characters if not present
    return f"{store_id[:2]}-{store_id[2:]}"

# Applying the function to the existing transactions_clean
transactions_clean['store_id'] = transactions_clean['store_id'].apply(format_state_code)
print(transactions_clean['store_id'].unique())

['FL-710' 'FL-412' 'FL-105' 'FL-330' 'FL-302' 'FL-445' 'FL-619' 'FL-287'
 'FL-239' 'FL-156' 'FL-501' 'FL-078']


In [39]:
# creating a function to standardize the unit_price
def format_unit_price(unit_price):
    if not isinstance(unit_price, str):
        return unit_price
    # remove $
    unit_price = unit_price.replace('$', '')
    # converting prices to numeric values
    unit_price = pd.to_numeric(unit_price, errors='coerce')
    return unit_price

# Applying the function to the existing transactions_clean
transactions_clean['unit_price'] = transactions_clean['unit_price'].apply(format_unit_price)
print(transactions_clean['unit_price'].unique())

[ 3.15  2.02 10.47 ... 30.48 24.39 14.24]


In [40]:
# Handling the timestamps - creating a standardized date and timestamp format and keeping existing rows that don't have timestamps
# Using pandas standardized datetime
def format_timestamps(df):
    df = df.copy()

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="mixed",
        errors="coerce"
    )

    return df

transactions_clean = format_timestamps(transactions_clean)
print(transactions_clean)



      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  
0             PEANUT BUTTER                Grocery         1        3.15  
1               CANNED SOUP      

In [41]:
# applying SKU logic already established to the transactions data
# keeping existing transactions SKU for now
transactions_clean["original_sku"] = (

    transactions_clean["sku"]
)

transactions_clean["canonical_sku"] = (
    transactions_clean["sku"].replace(SKU_MAP)
)
print(transactions_clean)

      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  \
0             PEANUT BUTTER                Grocery         1        3.15   
1               CANNED SOUP    

In [42]:
# doing the same process for product names - ensuring they match what was already established in the product
# cleaning the text using same conventions for products_clean
transactions_clean['product_name'] = (
    transactions_clean['product_name']
    .str.lower()
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('_', ' ', regex=False)
    .str.replace('pk', '-pack', regex=False)
    .str.replace('sz', 'size', regex=False)
    .str.replace(',', '', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
    .str.replace('Aa', 'AA', regex=False)
)
#verifying that cleaning produced the desired result
print(transactions_clean['product_name'].unique())
# mapping cleaned values to PRODUCT MAP logic already established
PRODUCT_MAP = {
    'Strawberry Pop-Tart': 'Pop-Tarts Strawberry',
    'Water - 24 Pack': 'Bottled Water 24-Pack',
    'Batteries AA 4-Pack': 'AA Batteries 4-Pack',
    'Board Game - Monopoly': 'Monopoly Board Game',
    'Streaming Device': 'Roku Streaming Stick',
    'Soup - Canned': 'Canned Soup',
    'Portable Phone Charger': 'Portable Charger',
    'Variety Pack - Chips': 'Chips Variety Pack',
    'Toilet Paper 12-Roll': 'Toilet Paper 12-Pack',
    'Baby Diapers - Size 3': 'Diapers Size 3',
    'Infant Formula': 'Baby Formula',
    'Tarp - Heavy Duty': 'Heavy Duty Tarp',
    'Flashlight Led': 'LED Flashlight',
    'Candles Emergency': 'Emergency Candles',
}

# Applying the dictionary replace on products_clean
transactions_clean['product_name'] = transactions_clean['product_name'].replace(PRODUCT_MAP)

# Printing values from products_clean to verify the result
print(transactions_clean['product_name'].unique())

# finally, checking to make sure transactions clean is looking accurate
print(transactions_clean)


['Peanut Butter' 'Canned Soup' 'Toilet Paper 12-Roll'
 'Roku Streaming Stick' 'Light Beer 24-Pack' 'Chips Variety Pack'
 'Tarp - Heavy Duty' 'Paper Towels 6-Pack' 'Emergency Candles'
 'Playing Cards' 'Bread - White' 'Portable Phone Charger' 'Paper Towels'
 'First Aid Kit' 'Pop-Tarts Strawberry' 'Flashlight - Led'
 'Peanut Butter 16Oz' 'Poptart-Strawberry' 'AA Batteries 4-Pack'
 'Strawberry Pop-Tart' 'Infant Formula' 'Water - 24 Pack'
 'Toilet Paper 12-Pack' 'Streaming Stick' 'Batteries AA 4-Pack'
 'Standard Playing Cards' 'Flashlight Led' 'Pop Tarts - Strawberry'
 'Portable Charger' 'Diapers Size 3' 'Heavy Duty Tarp' 'White Bread'
 'Bottled Water 24-Pack' 'Soup - Canned' 'Phone Charger - Portable'
 'Monopoly Board Game' 'Beer Light 24-Pack' 'Variety Pack - Chips'
 'Candles Emergency' 'Candles - Emergency' 'Hd Tarp'
 'Board Game - Monopoly' 'Streaming Device' 'Led Flashlight'
 'Baby Diapers - Size 3' 'Tp 12 Pack' 'Baby Formula' 'First-Aid Kit'
 'Paper Towels 6-Roll' 'Monopoly']
['Peanut

In [43]:
# applying a similar logic to make sure categories match up with those established in the product catalog
#Logic: stripping unnecessary descriptions + standardizing capitalization conventions
transactions_clean['category'] = (
    transactions_clean['category']
    .str.lower()
    .str.title()
)
# creating a category map
CATEGORY_MAP = {'Breakfast Foods': 'Breakfast',
           'Emergency Supplies': 'Emergency',
           'Alcoholic Beverages': 'Alcohol',
           'Grocery': 'Groceries',
           'Baby': 'Baby Products',
}
transactions_clean['category'] = transactions_clean['category'].replace(CATEGORY_MAP)
print(transactions_clean)

      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  \
0             Peanut Butter              Groceries         1        3.15   
1               Canned Soup    

In [44]:
# last step: converting quantity into a numeric value
def format_quantity(quantity):
    if not isinstance(quantity, str):
        return quantity
    # converting quantity to numeric values
    quantity = pd.to_numeric(quantity, errors='coerce')
    return quantity
print(transactions_clean)


      transaction_id store_id           timestamp            sku  \
0          TXN012636   FL-710 2024-09-13 09:48:13     PNUTBTR-16   
1          TXN000759   FL-412 2024-09-02 20:30:20      CNSOUP-01   
2          TXN003338   FL-105 2024-09-06 10:06:00  TOILETPAPER12   
3          TXN025020   FL-330 2024-09-30 06:29:08       ROKU-STD   
4          TXN015797   FL-302 2024-09-15 15:59:42        BEER-24   
...              ...      ...                 ...            ...   
25557      TXN020201   FL-105 2024-09-22 18:28:07      DIAPERS-3   
25558      TXN014835   FL-239 2024-09-15 08:47:08       CARDS-01   
25559      TXN002434   FL-239 2024-09-05 16:39:00    PAPERTOWEL6   
25560      TXN014739   FL-710 2024-09-14 17:08:35      CHRG-PORT   
25561      TXN010981   FL-105 2024-09-13 06:32:00        DIAP-S3   

               product_name               category  quantity  unit_price  \
0             Peanut Butter              Groceries         1        3.15   
1               Canned Soup    